# 第1周：用户空间深入

> **学习目标**：理解文件系统底层（inode/链接）、文件权限与 ACL、进程模型（/proc/信号/僵尸进程/daemon）、管道重定向与文件描述符、Shell 编程进阶（set -euo pipefail/trap/数组）、文本处理工具链（sed/awk/jq）

---


## 开篇：用户空间就是你的日常

如果你已经会用 `cd`、`ls`、`grep`、`find`，那恭喜你，你已经是一名合格的 Linux 用户了。但这一周我们将探索**表象之下的本质**：

- `ls -l` 的第一列那个数字是什么？ -- inode
- 为什么 `rm` 一个文件后，正在运行的进程还能继续读写它？ -- 文件描述符与引用计数
- `|` 管道到底做了什么？ -- 进程间通信
- 为什么有的脚本开头要写 `set -euo pipefail`？ -- 健壮 Shell 编程

这一周的内容偏底层，但每一条知识都会在未来排查问题时派上用场。我们不会停留在"怎么用"的层面，而是要深入理解"为什么这样工作"。

---

## Day 1：文件系统底层

### inode -- 文件名与数据的桥梁

你有没有想过：**文件名存在哪里？文件的内容存在哪里？它们之间怎么关联？**

答案是：文件名存在**目录**中，文件内容存在**数据块**中，而 **inode** 是连接它们的桥梁。
每个文件都有一个 inode，inode 中存储了文件的元数据（权限、大小、时间戳、数据块位置），但**不包含文件名**。文件名只存在于目录的条目中。

```
目录条目          inode          数据块
a.txt -> 12345 -> 权限/大小/指针 -> "hello\n"
```

**关键点**：
- 文件名和 inode 是一一对应的关系，一个文件可以有多个名字（硬链接）
- inode 存储元数据：权限、大小、时间戳、数据块位置
- `stat` 命令可以查看 inode 的全部信息
- `ls -i` 显示文件的 inode 号

### inode 耗尽

这是一个经典场景：`df -h` 显示磁盘还有空间，但创建文件时提示"No space left on device"。
原因可能是 inode 用完了！每个分区创建时决定了 inode 数量（每 16KB 空间一个 inode）如果存放了大量小文件（如 Git 仓库、Cache 目录），inode 可能先于空间耗尽。

```bash
df -i /data    # 查看 inode 使用率
df -h /data    # 查看空间使用率
```

### 硬链接 vs 软链接

| 特性 | 硬链接 | 软链接（符号链接） |
|------|--------|-------------------|
| 本质 | 同一个 inode 的另一个名字 | 存着指向目标路径的文本 |
| 跨文件系统 | 不行 | 可以 |
| 跨分区 | 不行 | 可以 |
| 链接目录 | 不行 | 可以 |
| 原文件删除后 | 仍可访问 | 断链 |
| ls -l 显示 | 第二列为链接数 | 箭头指向目标 |
| 大小 | 和原文件相同 | 路径字符串的长度 |

**为什么硬链接不能跨文件系统？** 因为 inode 号只在同一个文件系统内唯一。不同文件系统的两个文件可能恰好有相同的 inode 号，指向完全不同的数据。

**为什么硬链接不能链接目录？** 因为可能造成循环引用，导致遍历时死循环。但`.`（当前目录）和`..`（上级目录）是特殊的硬链接，由内核管理。


In [ ]:
# inode 与链接实验
! echo "=== 创建文件，观察 inode ==="
! echo 'Hello Linux Filesystem' > a.txt
! ls -li a.txt
! stat a.txt
echo ""
! echo "=== 硬链接实验 ==="
! ln a.txt hard.txt
! ls -li a.txt hard.txt  # inode 相同！
! echo "注意 ls -l 第二列的链接数："
! ls -l a.txt
echo ""
! echo "=== 软链接实验 ==="
! ln -s a.txt soft.txt
! ls -li a.txt soft.txt  # inode 不同！
! cat soft.txt
echo ""
! echo "=== 删除原文件后的区别 ==="
! rm a.txt
! cat hard.txt   # 还能读
! cat soft.txt   # 断链了
! ls -li hard.txt soft.txt
echo ""
! echo "=== inode 使用率 ==="
! df -i / 2>/dev/null || echo "查看根分区 inode 使用率"
echo ""
! echo "=== 链接数为 1 的普通文件 ==="
! touch /tmp/single.txt && ls -l /tmp/single.txt && rm /tmp/single.txt
echo ""
! echo "=== 目录的链接数 ==="
! ls -ld .  # 目录的链接数 = 子目录数 + 2
! mkdir -p /tmp/linktest/subdir
! ls -ld /tmp/linktest  # 有子目录后链接数增加
! rm -rf /tmp/linktest

---
## Day 2：文件权限与 ACL

### rwx 的含义 -- 文件和目录大不同

相同的 `rwx` 权限位，在文件和目录上含义完全不同。这是初学者最容易混淆的地方：

| 权限 | 文件 | 目录 |
|------|------|------|
| r | 读文件内容（cat/less） | 列出目录内容（ls） |
| w | 修改文件内容（echo >/vim） | 创建/删除/重命名文件 |
| x | 执行文件（./script） | 进入目录（cd），访问里面的文件 |

> **实战经验**：目录如果没有 x 权限，即使有 r 权限你也无法 cd 进去，`ls` 会报错。Web 服务器报 403 Forbidden 时，90% 的情况是目录缺少 x 权限。排查：`ls -ld /path/to/webroot` 检查目录权限。

### setuid / setgid / sticky bit

**setuid（4xxx）**：执行时以文件**所有者**的权限运行。
- `passwd` 命令：`-rwsr-xr-x`，普通用户通过它修改 /etc/shadow
- `sudo`：`---s--x--x`
- 原理：进程的 euid（有效用户 ID）临时变为文件所有者的 uid

**setgid（2xxx）**：执行时以文件所属组的权限运行。在目录上设置时，新建文件和子目录自动继承该目录的组。
实用场景：团队共享目录，所有人建的文件自动属于同一个组。

**sticky bit（1xxx）**：目录下只有文件所有者能删除自己的文件。
- `/tmp`：`drwxrwxrwt`，任何人都可以写，但只能删自己的文件
- 防止用户 A 删掉用户 B 的临时文件

### umask -- 控制默认权限

```
文件默认 666 - umask = 实际权限
目录默认 777 - umask = 实际权限
umask 022 -> 文件 644 (rw-r--r--), 目录 755 (rwxr-xr-x)
umask 077 -> 文件 600 (rw-------), 目录 700 (rwx------)
```

安全敏感场景下（如 Web 服务器），umask 应该设为 027 或 077。

### ACL -- 更精细的权限控制

当 ugo 三段式不够用时（比如需要给 5 个不同用户不同的权限），用 ACL：

```bash
setfacl -m u:alice:rw myfile    # 给 alice 读写
setfacl -m g:devs:rx myfile     # 给 devs 组读+执行
setfacl -m o::--- myfile        # 其他用户无权限
setfacl -x u:alice myfile       # 移除 alice 的 ACL
setfacl -b myfile               # 移除所有 ACL
getfacl myfile                  # 查看 ACL
```

设置 ACL 后 `ls -l` 的权限位末尾会多一个 `+`，提示你这里不只是简单的 ugo 权限。


In [ ]:
# 文件权限实验
! echo "=== 查看特殊权限文件 ==="
! ls -l $(which passwd) 2>/dev/null
! ls -l $(which sudo) 2>/dev/null
! ls -ld /tmp
echo ""
! echo "=== setgid 目录演示 ==="
! mkdir -p /tmp/sgid_test
! chmod g+s /tmp/sgid_test  # 设置 setgid
! ls -ld /tmp/sgid_test      # 权限位应该有 s
! rm -rf /tmp/sgid_test
echo ""
! echo "=== umask 实验 ==="
! umask 022
! touch /tmp/f_022 && ls -l /tmp/f_022
! rm -f /tmp/f_022
! umask 077
! touch /tmp/f_077 && ls -l /tmp/f_077
! rm -f /tmp/f_077
! umask 022
echo ""
! echo "=== ACL 实验 ==="
! touch /tmp/acl_test
! setfacl -m u:root:rwx /tmp/acl_test
! setfacl -m g:root:rx /tmp/acl_test
! echo "ACL 设置后："
! getfacl /tmp/acl_test
! echo "ls -l 显示（注意末尾的 +）："
! ls -l /tmp/acl_test
! setfacl -b /tmp/acl_test  # 移除所有 ACL
! rm /tmp/acl_test
echo ""
! echo "=== chmod 数字方式速记 ==="
! echo "rwx=7, rw-=6, r-x=5, r--=4"
! echo "chmod 755 = rwxr-xr-x"
! echo "chmod 4755 = rwsr-xr-x（setuid）"
! echo "chmod 2755 = rwxr-sr-x（setgid）"
! echo "chmod 1755 = rwxr-xr-t（sticky）"

---
## Day 3：进程深入

### /proc -- 进程的镜子

`/proc` 是一个**虚拟文件系统**，内核把进程信息暴露成文件和目录。每个运行的进程都在 /proc/<pid>/ 下对应一个目录：

```
/proc/<pid>/cmdline    # 启动命令
/proc/<pid>/environ    # 环境变量
/proc/<pid>/fd/        # 文件描述符
/proc/<pid>/status     # 状态、内存、进程名
/proc/<pid>/maps       # 内存映射
/proc/<pid>/limits     # 资源限制
/proc/<pid>/cwd        # 当前工作目录
/proc/<pid>/root       # 根目录
/proc/<pid>/oom_score  # OOM 评分
```

> **实用技巧**：`ls -l /proc/<pid>/fd` 能清楚看到进程打开了哪些文件。如果你怀疑 Docker 容器里的 pid 与宿主机 pid 不一致，用 nsenter 进入容器的命名空间查看。

### fork / exec -- 进程的诞生

**fork**：复制当前进程，创建子进程。子进程获得父进程的完整副本（包括内存、fd、环境变量）。
**exec**：替换当前进程的程序镜像（加载新的二进制文件运行）。
返回码为 -1 时表示 exec 失败（如文件不存在、权限不够）。

```
Shell (bash)
  |
  +-- fork() --> 子进程（完全复制 bash）
  |                |
  |                +-- exec("/bin/ls") --> ls 进程
  |
  +-- wait() <-- 等待子进程结束
```

Docker 的 entrypoint 也是这个原理 -- 容器启动时 PID 1 执行 fork+exec 运行你的程序。

### 信号 -- 进程间最简单的通信

信号是异步的，内核可以随时打断一个进程的执行并向它发送信号。

```bash
kill -l                    # 列出所有信号（Linux 上有 64 个）
kill -TERM <pid>           # 发送 SIGTERM（优雅终止）
kill -KILL <pid>           # 发送 SIGKILL（强制杀掉）
kill -HUP <pid>            # 发送 SIGHUP（重载配置）
```

| 信号 | 编号 | 默认行为 | 能否捕获 | 用途 |
|------|------|----------|---------|------|
| SIGHUP | 1 | 终止 | 能 | 重载配置（nginx -s reload） |
| SIGINT | 2 | 终止 | 能 | 键盘 Ctrl+C |
| SIGQUIT | 3 | 终止+core | 能 | Ctrl+\ |
| SIGKILL | 9 | 终止 | **不能** | 强制杀掉 |
| SIGTERM | 15 | 终止 | 能 | 优雅停止（默认） |
| SIGUSR1 | 10 | 终止 | 能 | 用户自定义，常用于日志重开 |
| SIGCHLD | 17 | 忽略 | 能 | 子进程退出时发给父进程 |

### 僵尸进程 vs 孤儿进程

- **僵尸进程**：子进程已退出但父进程没有调用 wait()。ps 显示状态为 Z。
  僵尸进程已释放了所有资源，只保留进程表中的一个条目。大量僵尸进程会耗尽系统进程数上限。必须靠父进程 wait() 或父进程退出才能清理。
- **孤儿进程**：父进程先于子进程退出。init（PID 1）会收养这些子进程并负责回收。

### Daemon -- 守护进程

守护进程的共同特征：
1. 脱离终端（没有 stdin/stdout/stderr 连接到终端）
2. 调用 setsid() 创建新会话
3. 工作目录改为 /
4. 关闭所有不必要的文件描述符
5. 重定向 stdin/stdout/stderr 到 /dev/null

在现代 Linux 中，systemd 帮我们做 daemonization，不需手动实现。
`Type=simple` 或 `Type=forking` 控制 systemd 如何识别 daemon 就绪。


In [ ]:
# 进程实验
! echo "=== 探索 /proc/self ==="
! ls /proc/self/fd 2>/dev/null
! echo "当前 Shell PID: $$"
! echo "当前进程名: $(cat /proc/$$/comm)"
! cat /proc/$$/status | head -12
echo ""
! echo "=== 查看进程资源限制 ==="
! cat /proc/$$/limits | head -8
echo ""
! echo "=== 信号列表 ==="
! kill -l | head -8
! echo "... 共 $(kill -l 2>/dev/null | wc -w) 个信号"
echo ""
! echo "=== 信号实验 ==="
! sleep 30 &
! SLEEPPID=$!
! echo "启动 sleep 进程 PID=$SLEEPPID"
! kill -TERM $SLEEPPID 2>/dev/null; echo "发送 SIGTERM"
! sleep 1; ps -p $SLEEPPID 2>/dev/null || echo "进程已终止"
echo ""
! echo "=== 僵尸进程演示 ==="
! python3 << 'PYEOF'
import os, time
pid = os.fork()
if pid == 0:
    print(f"[child] PID={os.getpid()} exiting now")
    os._exit(0)
else:
    print(f"[parent] PID={os.getpid()}, child PID={pid}")
    print("check Z process: ps aux | grep Z")
    time.sleep(15)
    print("[parent] cleaning up zombie")
    os.wait()
    print("[parent] done")
PYEOF
echo ""
! echo "=== PID 1 (systemd) 的文件描述符 ==="
! ls /proc/1/fd/ 2>/dev/null | head -10 || echo "（无权限，这很正常）"

---
## Day 4：管道、重定向、文件描述符

### 3 个标准文件描述符

每个进程启动时，内核自动打开三个文件描述符：

| fd | 名称 | 默认连接 | 重定向写法 |
|----|------|----------|-----------|
| 0 | stdin | 键盘/输入 | `< file` |
| 1 | stdout | 终端/输出 | `> file` `>> file` |
| 2 | stderr | 终端/错误 | `2> file` `2>> file` |

### 重定向运算符大全

| 语法 | 含义 | 示例 |
|------|------|------|
| `>` | stdout 覆盖写入 | `echo hi > file` |
| `>>` | stdout 追加写入 | `echo hi >> file` |
| `2>` | stderr 覆盖写入 | `cmd 2> err.log` |
| `&>` | stdout+stderr | `cmd &> all.log` |
| `2>&1` | stderr 合并到 stdout | `cmd > out 2>&1` |
| `1>&2` | stdout 合并到 stderr | `echo err 1>&2` |
| `<>` | 读写方式打开 | `exec 3<> file` |

### 管道 `|`

管道把前一个命令的 stdout 连接到后一个命令的 stdin。**每个管道符启动一个子进程**。

```
ps aux | grep python | head -5
  进程1     进程2      进程3
  stdout -> stdin  stdout -> stdin
```

### 命名管道（FIFO）

普通管道只能在父子进程或相关进程间通信。命名管道通过文件系统实现**任意进程间通信**：

```bash
mkfifo mypipe
# 终端 A: cat mypipe    （阻塞等待，直到有数据写入）
# 终端 B: echo hi > mypipe  （写入数据，A 收到后退出）
```

### 进程替换（Process Substitution）

Bash 特有语法，将命令的输出伪装成文件：

```bash
diff <(ls dir1) <(ls dir2)              # 比较两个目录
source <(echo 'export FOO=bar')          # 从命令输出 source
while read line; do ...; done < <(cmd)   # 逐行处理命令输出
```


In [ ]:
# 管道与重定向实验
! echo "=== 基础重定向 ==="
! echo "stdout 消息" > /tmp/redir.txt
! cat /tmp/redir.txt
! echo "追加一行" >> /tmp/redir.txt
! cat /tmp/redir.txt
echo ""
! echo "=== 标准错误重定向 ==="
! ls /nonexistent 2> /tmp/err.txt
! cat /tmp/err.txt
echo ""
! echo "=== stdout+stderr 合并 ==="
! (echo "out"; ls /nonexistent) &> /tmp/all.txt
! cat /tmp/all.txt
echo ""
! echo "=== tee 同时输出到终端和文件 ==="
! echo "Hello tee" | tee /tmp/tee_test.txt
echo ""
! echo "=== 命名管道 ==="
! mkfifo /tmp/myfifo 2>/dev/null || true
! ls -la /tmp/myfifo
! echo "两个终端: cat /tmp/myfifo | echo hi > /tmp/myfifo"
! rm -f /tmp/myfifo
echo ""
! echo "=== exec 操作文件描述符 ==="
! exec 3> /tmp/fd_test.txt
! echo "通过 fd 3 写入" >&3
! exec 3>&-
! cat /tmp/fd_test.txt
echo ""
! echo "=== /dev/null 比特黑洞 ==="
! echo "看不见我" > /dev/null
! ls /nonexistent 2> /dev/null  # 错误也被吞了
echo ""
! echo "=== 进程替换 ==="
! mkdir -p /tmp/dir1 /tmp/dir2
! touch /tmp/dir1/file_a /tmp/dir2/file_b
! diff <(ls /tmp/dir1) <(ls /tmp/dir2) || true
! rm -rf /tmp/dir1 /tmp/dir2 /tmp/redir.txt /tmp/err.txt /tmp/all.txt /tmp/tee_test.txt /tmp/fd_test.txt

---
## Day 5：Shell 编程进阶

### 变量扩展 -- ${} 的常见用法

| 语法 | 含义 | 示例 |
|------|------|------|
| `${var:-default}` | 未设置时用默认值 | `${PORT:-8080}` |
| `${var:=default}` | 未设置时设默认值 | `${PORT:=8080}` |
| `${var:+alt}` | 已设置时用替代值 | `${DEBUG:+'-v'}` |
| `${var:?error}` | 未设置时报错退出 | `${DB_HOST:?required}` |
| `${var#pat}` | 前缀删最短匹配 | 取文件名 |
| `${var##pat}` | 前缀删最长匹配 | `basename` |
| `${var%pat}` | 后缀删最短匹配 | 去扩展名 |
| `${var/old/new}` | 替换第一个匹配 | |

### 函数

```bash
log_info() {
    local msg="$1"  # local 是关键！不加就被全局可见
    echo "[INFO] $(date) $msg"
}
```

Shell 函数的特殊性：参数传递和 C/Java 不同，函数内部用 `$1` `$2` 访问参数，
`$?` 获取返回值，`return` 返回数字（0=成功，非0=失败）。

### 函数 vs 脚本

函数在当前 Shell 进程中执行，可以修改当前 Shell 的变量。
脚本在子 Shell 中执行，不能修改父 Shell 的变量（除非 source）。

### 错误处理三件套

每个正式 Shell 脚本开头都应该有这行：`set -euo pipefail`

| 选项 | 作用 | 经验 |
|------|------|------|
| `-e` | 任何命令失败就终止 | 防止错误被忽略 |
| `-u` | 使用未定义变量时报错 | 防止拼写错误 |
| `-o pipefail` | 管道任一命令失败即失败 | 防止管道吞掉错误 |

但注意：`-e` 在某些情况不会触发（如 if 条件中的命令、&& 或 || 后面的命令）。

### trap -- 优雅清理

```bash
cleanup() {
    echo "清理临时文件..."
    rm -rf /tmp/myapp_*
}
trap cleanup EXIT       # 脚本无论如何退出都会执行
trap 'echo Ctrl+C; exit 1' INT  # Ctrl+C 时执行
trap '' SIGTERM         # 忽略 SIGTERM
```

### 数组

```bash
arr=(apple banana cherry)
echo ${arr[0]}          # apple
echo ${arr[@]}          # apple banana cherry
echo ${#arr[@]}         # 3
for item in "${arr[@]}"; do echo "$item"; done
```

### 调试

```bash
bash -x script.sh       # 打印每条执行命令（前面有 +）
set -x                  # 在脚本中开启调试
set +x                  # 关闭调试
```
调试时配合 `PS4='$LINENO: '` 可以让每行显示行号，更好定位。


In [ ]:
# Shell 编程进阶实验
! echo "=== 变量扩展演示 ==="
! echo "PORT 未设置: ${PORT:-8080}"
! PORT=3000
! echo "PORT 已设置: ${PORT:-8080}"
! unset PORT
echo ""
! echo "=== 字符串操作 ==="
! FILE="/home/user/data/file.txt"
! echo "全路径: $FILE"
! echo "取文件名: ${FILE##*/}"
! echo "去扩展名: ${FILE%.*}"
! echo "替换 txt -> md: ${FILE/txt/md}"
echo ""
! echo "=== 数组操作 ==="
! SERVERS=(web01 web02 db01 cache01)
! echo "服务器列表: ${SERVERS[@]}"
! echo "数量: ${#SERVERS[@]}"
! for srv in "${SERVERS[@]}"; do
!     echo "  处理: $srv"
! done
echo ""
! echo "=== 备份脚本 ==="
! cat << 'SCRIPT' > /tmp/backup_demo.sh
#!/bin/bash
set -euo pipefail
BACKUP_DIR="/tmp/demo_backups"
log() { echo "[$(date +%H:%M:%S)] $*"; }
cleanup() { log "cleanup temp files"; }
trap cleanup EXIT
mkdir -p "$BACKUP_DIR"
log "backing up /tmp..."
tar -czf "${BACKUP_DIR}/backup_$(date +%Y%m%d).tar.gz" /tmp/ 2>/dev/null || log "tar done"
log "backup complete: $(ls $BACKUP_DIR)"
SCRIPT
! chmod +x /tmp/backup_demo.sh
! bash /tmp/backup_demo.sh
! rm -rf /tmp/backup_demo.sh /tmp/demo_backups
echo ""
! echo "=== 调试模式 ==="
! bash -c 'set -x; echo hello; x=world; echo $x' 2>&1 | head -4

---
## Day 6：文本处理工具链

### sed -- 流编辑器

sed 的核心是读一行 -> 执行命令 -> 输出。

```bash
sed 's/old/new/g' file             # 全局替换
sed -i 's/old/new/g' file          # 原地修改
sed -n '/pattern/p' file           # 只打印匹配的行
sed '/start/,/end/d' file          # 删除区间行
sed '3,5d' file                    # 删除 3-5 行
sed 's/^/  /' file                 # 每行前加两个空格缩进
sed -n '10,20p' file               # 打印 10-20 行
```

### awk -- 字段处理大师

awk 自动把每行拆成字段：`$1`, `$2`, ..., `$NF`（最后一列）。

```bash
awk '{print $1, $NF}' file           # 第1列和最后一列
awk '$3 > 100 {print}' file          # 条件过滤
awk '{sum+=$3} END{print sum}'       # 求和
awk 'NR>1 && NR<10' file             # 第 2-9 行
awk -F: '{print $1}' /etc/passwd     # 指定分隔符
awk '!seen[$1]++' file               # 去重
```

### sort + uniq -- 统计利器

```bash
sort file | uniq -c | sort -rn       # 频率统计（经典）
sort -t: -k3 -n /etc/passwd          # 按第3列数字排序
sort -u file                         # 排序并去重
```

### xargs -- 连接管道和参数

```bash
find . -name '*.py' | xargs grep 'TODO'    # 查找 TODO
cat urls.txt | xargs -P 4 curl -s          # 并行下载
find . -name '*.log' -mtime +7 | xargs rm -f  # 删旧日志
```

### jq -- JSON 处理

```bash
jq '.key' file.json                    # 读取 key
jq -r '.key' file.json                 # 原始字符串输出
jq '.[] | {name, ip}' file             # 提取字段
jq '. | length' file                   # 数组长度
```


In [ ]:
# 文本处理实验
! echo "=== 准备测试数据 ==="
! cat << 'DATA' > /tmp/access.log
192.168.1.1 GET /index.html 200 2326
192.168.1.2 GET /api/users 200 1234
192.168.1.1 GET /index.html 200 2326
192.168.1.3 POST /api/login 401 56
192.168.1.1 GET /images/logo.png 200 45678
192.168.1.2 GET /api/users 200 1234
192.168.1.4 GET /index.html 200 2326
192.168.1.3 POST /api/data 500 120
192.168.1.5 GET /favicon.ico 404 18
DATA
echo ""
! echo "=== sed 替换 ==="
! sed 's/GET/HTTP_GET/g' /tmp/access.log | head -3
echo ""
! echo "=== awk: Top 5 IP ==="
! awk '{print $1}' /tmp/access.log | sort | uniq -c | sort -rn | head -5
echo ""
! echo "=== awk: 状态码统计 ==="
! awk '{print $3}' /tmp/access.log | sort | uniq -c | sort -rn
echo ""
! echo "=== awk: 只显示 4xx/5xx 错误 ==="
! awk '$3 ~ /^[45]/' /tmp/access.log
echo ""
! echo "=== awk: 总传输字节 ==="
! awk '{sum+=$4} END{print "总字节:", sum}' /tmp/access.log
echo ""
! echo "=== awk: 各 IP 传输量 ==="
! awk '{bytes[$1]+=$4} END{for(ip in bytes) print ip, bytes[ip]}' /tmp/access.log | sort -k2 -rn
echo ""
! echo "=== xargs 示例 ==="
! echo "file1.txt file2.txt" | xargs -n1 echo "处理:"
echo ""
! echo "=== jq 演示 ==="
! echo '{"items":[{"n":"web01","ip":"10.0.0.1"},{"n":"db01","ip":"10.0.0.2"}]}' | jq '.items[] | {host:.n, addr:.ip}'
! rm -f /tmp/access.log

---
## Day 7：第一周综合练习

### 任务：系统巡检脚本

写一个系统巡检脚本 `syscheck.sh`，收集并格式化输出以下信息：

```
系统信息：主机名、内核版本、运行时间
资源使用：CPU 负载、内存使用（已用/总量/百分比）、磁盘使用率、inode 使用率
Top 进程：CPU 最高的 5 个进程、内存最高的 5 个进程
网络状况：监听端口列表、活跃连接数
安全：SSH 失败登录次数
```

### 要求

1. 脚本开头必须包含 `set -euo pipefail`
2. 至少使用 2 个函数，如 `print_section()` 和 `get_info()`
3. 使用 `trap` 处理退出清理
4. 使用 `awk` 处理命令输出
5. 对没有权限读取的信息给出友好提示，而非直接崩溃
6. 输出格式清晰，可以使用颜色或 ASCII 边框

### 扩展挑战

完成基础版本后，尝试以下扩展：
- 增加 `nvidia-smi` 检测（GPU 信息）
- 增加 Docker 容器运行状态检查
- 输出改为 JSON 格式，配合 `jq` 处理
- 配置 cron 或 systemd timer 每 15 分钟运行一次，追加输出到日志文件中
- 用 awk 分析日志文件：找出 CPU 负载超过 2.0 的时间点


In [ ]:
# 综合练习：系统巡检脚本
! cat << 'SCRIPT' > /tmp/syscheck.sh
#!/bin/bash
set -euo pipefail
RED='\033[0;31m'; GREEN='\033[0;32m'; CYAN='\033[0;36m'
YELLOW='\033[1;33m'; NC='\033[0m'
TEMPFILE=$(mktemp)
cleanup() { rm -f "$TEMPFILE"; echo; }
trap cleanup EXIT
section() { printf "\n${CYAN}================ %s ================${NC}\n" "$1"; }
info() { printf "${GREEN}%-18s${NC} %s\n" "$1:" "$2"; }
warn() { printf "${YELLOW}%-18s${NC} %s\n" "$1:" "$2"; }
section "系统信息"
info "主机名" "$(hostname 2>/dev/null || echo N/A)"
info "内核" "$(uname -r)"
info "运行时间" "$(uptime -p 2>/dev/null | sed 's/up //' || echo N/A)"
section "CPU 负载"
uptime | awk -F'load average:' '{print "  Load: " $2}'
section "内存"
free -h | awk '/^Mem:/ {printf "  已用: %s / 总量: %s (%.1f%%)\n", $3, $2, ($3/$2)*100}'
section "磁盘"
df -h 2>/dev/null | awk 'NR>1 {printf "  %-15s %s\n", $1, $5}' || warn "磁盘" "无法读取"
section "Inode"
df -i 2>/dev/null | awk 'NR>1 {printf "  %-15s %s\n", $1, $5}' | sort -k2 -rn | head -3
section "Top 5 CPU"
ps aux --sort=-%cpu 2>/dev/null | awk 'NR>1&&NR<=6 {printf "  PID=%-6s %-25s CPU=%s%%\n", $2, $11, $3}' || warn "进程" "权限不足"
section "Top 5 内存"
ps aux --sort=-%mem 2>/dev/null | awk 'NR>1&&NR<=6 {printf "  PID=%-6s %-25s MEM=%s%%\n", $2, $11, $4}' || true
section "监听端口"
if command -v ss &>/dev/null; then
  ss -tlnp 2>/dev/null | awk 'NR>1 {print "  " $4}' | head -8 || warn "端口" "无权限"
fi
section "Connections"
ss -tan 2>/dev/null | tail -n +2 | wc -l | xargs -I{} echo "  Active: {} connections"
section "安全"
for f in /var/log/auth.log /var/log/secure; do
  [ -r "$f" ] && grep -c "Failed password" "$f" 2>/dev/null | xargs -I{} echo "  SSH失败登录: {} 次" && break
done || warn "安全" "日志不可访问"
echo -e "\n${GREEN}巡检完成${NC}"
SCRIPT
! chmod +x /tmp/syscheck.sh
! echo "=== 运行系统巡检脚本 ==="
! bash /tmp/syscheck.sh 2>/dev/null || echo "部分信息因权限不足无法显示（这是正常的）"
echo ""
! cat << 'CHECK'
=== 自检清单 ===
[ ] 运行并逐行理解输出含义
[ ] 增加 GPU 检测 (nvidia-smi)
[ ] 增加 Docker 容器状态检查
[ ] 修改输出为 JSON 格式
[ ] 配置 cron: */15 * * * * /tmp/syscheck.sh >> /var/log/syscheck.log
CHECK

---
## 第1周总结

### 核心概念

| 概念 | 一句话 |
|------|--------|
| **inode** | 存储文件元数据，连接文件名与数据 |
| **硬链接** | 同一个 inode 的多个名字，不能跨文件系统 |
| **软链接** | 存目标路径的快捷方式，可跨文件系统 |
| **setuid** | 执行时以文件所有者身份运行（passwd/sudo） |
| **sticky bit** | 目录下只能删自己的文件（/tmp） |
| **umask** | 控制新建文件和目录的默认权限 |
| **ACL** | setfacl/getfacl，比 ugo 更细粒度 |
| **/proc** | 虚拟文件系统，暴露进程运行时信息 |
| **fork/exec** | fork 复制进程，exec 替换程序镜像 |
| **信号** | SIGTERM(15) 优雅 / SIGKILL(9) 强制 |
| **僵尸进程** | 子进程已退出但父进程没 wait（状态 Z） |
| **文件描述符** | 0=stdin, 1=stdout, 2=stderr |
| **set -euo pipefail** | Shell 脚本健壮三件套 |
| **trap** | 退出时自动执行清理操作 |

### 本周命令速查

```bash
ls -li file             # 查看 inode 号
ln src dst              # 硬链接
ln -s src dst           # 软链接
stat file               # 查看 inode 详细信息
getfacl / setfacl       # ACL 操作
umask                   # 默认权限掩码
ls /proc/<pid>/fd       # 进程文件描述符
kill -l                 # 列出所有信号
mkfifo                  # 创建命名管道
diff <(c1) <(c2)       # 进程替换
sed 's/old/new/g'      # 流编辑替换
awk '{print $1}'        # 字段提取
sort | uniq -c | sort -rn  # 频率统计经典组合
jq '.' file.json        # JSON 处理
```

### 下一步

第二周进入**系统管理**—— systemd（PID 1）、用户组与 sudo、PAM 认证、LVM 存储、包管理。
从"会写脚本"升级到"能管理服务器"。
